#### Objective's -

1. Understand and implement of DES round key generation.
2. Execution of DES algorithm for encryption and decryption of digital information. 

In [1]:
# ------------------ HEX <-> BIN ------------------
def hex_to_bin(s):
    return bin(int(s, 16))[2:].zfill(len(s) * 4)

def bin_to_hex(s):
    return hex(int(s, 2))[2:].upper().zfill(len(s) // 4)

# ------------------ PERMUTATION ------------------
def permute(data, table):
    return ''.join(data[i - 1] for i in table)

# ------------------ SHIFT ------------------
def shift_left(data, n):
    return data[n:] + data[:n]

# ------------------ XOR ------------------
def xor(a, b):
    return ''.join('0' if i == j else '1' for i, j in zip(a, b))

# ------------------ TABLES ------------------
IP = [58,50,42,34,26,18,10,2,
      60,52,44,36,28,20,12,4,
      62,54,46,38,30,22,14,6,
      64,56,48,40,32,24,16,8,
      57,49,41,33,25,17,9,1,
      59,51,43,35,27,19,11,3,
      61,53,45,37,29,21,13,5,
      63,55,47,39,31,23,15,7]

FP = [40,8,48,16,56,24,64,32,
      39,7,47,15,55,23,63,31,
      38,6,46,14,54,22,62,30,
      37,5,45,13,53,21,61,29,
      36,4,44,12,52,20,60,28,
      35,3,43,11,51,19,59,27,
      34,2,42,10,50,18,58,26,
      33,1,41,9,49,17,57,25]

EP = [32,1,2,3,4,5,4,5,6,7,8,9,
      8,9,10,11,12,13,12,13,14,15,16,17,
      16,17,18,19,20,21,20,21,22,23,24,25,
      24,25,26,27,28,29,28,29,30,31,32,1]

P4 = [16,7,20,21,29,12,28,17,
      1,15,23,26,5,18,31,10,
      2,8,24,14,32,27,3,9,
      19,13,30,6,22,11,4,25]

# PC-1 (Parity Drop)
PC1 = [57,49,41,33,25,17,9,
       1,58,50,42,34,26,18,
       10,2,59,51,43,35,27,
       19,11,3,60,52,44,36,
       63,55,47,39,31,23,15,
       7,62,54,46,38,30,22,
       14,6,61,53,45,37,29,
       21,13,5,28,20,12,4]

# PC-2 (Compression)
PC2 = [14,17,11,24,1,5,3,28,
       15,6,21,10,23,19,12,4,
       26,8,16,7,27,20,13,2,
       41,52,31,37,47,55,30,40,
       51,45,33,48,44,49,39,56,
       34,53,46,42,50,36,29,32]

SHIFT_TABLE = [1, 1, 2, 2, 2, 2, 2, 2,
               1, 2, 2, 2, 2, 2, 2, 1]

# ------------------ KEY GENERATION ------------------
def generate_keys(key):
    key = permute(key, PC1)
    left, right = key[:28], key[28:]
    round_keys = []

    for i in range(16):
        left = shift_left(left, SHIFT_TABLE[i])
        right = shift_left(right, SHIFT_TABLE[i])
        combined = left + right
        round_key = permute(combined, PC2)
        round_keys.append(round_key)

    return round_keys

# ------------------ SBOX (ONLY S1 for demo, rest similar) ------------------
SBOX = [
    [[14,4,13,1,2,15,11,8,3,10,6,12,5,9,0,7],
     [0,15,7,4,14,2,13,1,10,6,12,11,9,5,3,8],
     [4,1,14,8,13,6,2,11,15,12,9,7,3,10,5,0],
     [15,12,8,2,4,9,1,7,5,11,3,14,10,0,6,13]]
] * 8  # reuse for simplicity

def sbox_substitution(data):
    result = ""
    for i in range(8):
        block = data[i*6:(i+1)*6]
        row = int(block[0] + block[5], 2)
        col = int(block[1:5], 2)
        val = SBOX[i][row][col]
        result += bin(val)[2:].zfill(4)
    return result

# ------------------ DES FUNCTION ------------------
def des_function(right, key):
    expanded = permute(right, EP)
    xored = xor(expanded, key)
    sbox = sbox_substitution(xored)
    return permute(sbox, P4)

# ------------------ ENCRYPT ------------------
def des_encrypt(pt, keys):
    pt = permute(pt, IP)
    left, right = pt[:32], pt[32:]

    for i in range(16):
        temp = right
        right = xor(left, des_function(right, keys[i]))
        left = temp

    combined = right + left
    return permute(combined, FP)

# ------------------ DECRYPT ------------------
def des_decrypt(ct, keys):
    return des_encrypt(ct, keys[::-1])

# ------------------ MAIN ------------------
plaintext_hex = input("Enter Plaintext (16 hex): ").upper()
key_hex = input("Enter Key (16 hex): ").upper()

plaintext_bin = hex_to_bin(plaintext_hex)
key_bin = hex_to_bin(key_hex)

round_keys = generate_keys(key_bin)

cipher_bin = des_encrypt(plaintext_bin, round_keys)
cipher_hex = bin_to_hex(cipher_bin)

decrypted_bin = des_decrypt(cipher_bin, round_keys)
decrypted_hex = bin_to_hex(decrypted_bin)

# ------------------ OUTPUT ------------------
print("\n--- INPUT ---")
print("Plaintext HEX :", plaintext_hex)
print("Plaintext BIN :", plaintext_bin)

print("\n--- KEY ---")
print("Key HEX :", key_hex)
print("Key BIN :", key_bin)

print("\n--- CIPHER ---")
print("Cipher HEX :", cipher_hex)
print("Cipher BIN :", cipher_bin)

print("\n--- DECRYPTED ---")
print("Decrypted HEX :", decrypted_hex)
print("Decrypted BIN :", decrypted_bin)


--- INPUT ---
Plaintext HEX : AABB09182736CCDD
Plaintext BIN : 1010101010111011000010010001100000100111001101101100110011011101

--- KEY ---
Key HEX : AABB09182736CCDD
Key BIN : 1010101010111011000010010001100000100111001101101100110011011101

--- CIPHER ---
Cipher HEX : 6964B3E893AEFE0C
Cipher BIN : 0110100101100100101100111110100010010011101011101111111000001100

--- DECRYPTED ---
Decrypted HEX : AABB09182736CCDD
Decrypted BIN : 1010101010111011000010010001100000100111001101101100110011011101
